# 1. Import Ntuple and DecayHash

In [ ]:
import root_pandas
#import uproot
import decayHash
import basf2 as b2
from decayHash import DecayHashMap
import numpy as np
import ROOT
import pandas

In [6]:
# read continuum samples
filename='../exp7_10_14b_qq_tau_1/generic_MC_e.root'
df_cont = root_pandas.read_root(filename,key='B0')

# Apply 95% efficiency cuts to reduce data size
df_cont_cut=df_cont.query('D_vtxReChi2<13 and B0_vtxReChi2<14 and 5.03<B0_roeMbc_my_mask and -3.5<B0_roeDeltae_my_mask<0.5 and \
                4.65<B0_CMS2_weMbc and -2.2<B0_CMS0_weDeltae<0.5 and abs(B0_roeCharge_my_mask)<3 and \
                -3<B0_deltaE<-1 and e_CMS_p>0.2 and 0<=B0_vetoeID<0.9 and B0_vetomuID<0.9').copy()
df_cont_cut['B0_isSignal'] = df_cont_cut['B0_isSignal'].fillna(-1)
df_cont_cut['D_isSignal'] = df_cont_cut['D_isSignal'].fillna(-1)
df_cont_cut['e_isSignal'] = df_cont_cut['e_isSignal'].fillna(-1)
df_cont_cut['B0_mcPDG'] = df_cont_cut['B0_mcPDG'].fillna(0)

df_cont_cut['DecayMode'] = 'continuum'

df_cont_cut.to_root('../exp7_10_14b_qq_tau_1/exp7_10_14b_e_qq_tau_cut.root', key='B0')

/cvmfs/belle.cern.ch/el7/externals/v01-10-02/Linux_x86_64/common/lib/python3.8/site-packages/root_numpy/_tree.py:575: UserWarning: converter for dtype('O') is not implemented (skipping)
  cobj = _librootnumpy.array2tree_toCObj(arr, name=name, tree=incobj)


In [8]:
# read BBbar samples
filename='generic_MC_e.root'
decayhash='hashmap_generic_MC_e.root'
df_B = root_pandas.read_root(filename,key='B0')
#hashmap = DecayHashMap(decayhash, removeRadiativeGammaFlag=False)
hashmap2 = DecayHashMap(decayhash, removeRadiativeGammaFlag=True)

# 2. Add DecayMode column to the Ntuple

In [76]:
from collections import OrderedDict

def found(modes,row):
    for mode in modes:
        decaytree = ROOT.Belle2.DecayTree(mode)
        if hashmap2.get_original_decay(row["B0_DecayHash"],row["B0_DecayHashEx"]).find_decay(decaytree):
            return True
    return False

def decay_mode(row):
    for name,modes in mode_dict.items():
        if found(modes,row):
            return name
    return 'bkg' # other_B_decay (hadronic) or hadronic_tau or other_D_decay or charged_B or continuum

# the order of keys might be important, try to keep the muon modes at the bottom for e reconstruction
# the e modes will be kept at the bottom for a muon reconstruction
mode_dict = OrderedDict()
mode_dict['sig_D_tau_nu']=['511 (-> -411 (-> 321 -211 -211) -15 (-> -11 12 -16) 16)',
                           '-511 (-> 411 (-> -321 211 211) 15 (-> 11 -12 16) -16)',
                           '511 (-> -411 (-> 321 -211 -211) -15 (-> -13 14 -16) 16)',
                           '-511 (-> 411 (-> -321 211 211) 15 (-> 13 -14 16) -16)']

mode_dict['sig_D_e_nu']=['511 (-> -411 (-> 321 -211 -211) -11 12)',
                         '-511 (-> 411 (-> -321 211 211) 11 -12)']

mode_dict['sig_Dst_tau_nu']=['511 (-> -413 (-> -411 (-> 321 -211 -211) 111) -15 (-> -11 12 -16) 16)',
                             '-511 (-> 413 (-> 411 (-> -321 211 211) 111) 15 (-> 11 -12 16) -16)',
                             '511 (-> -413 (-> -411 (-> 321 -211 -211) 111) -15 (-> -13 14 -16) 16)',
                             '-511 (-> 413 (-> 411 (-> -321 211 211) 111) 15 (-> 13 -14 16) -16)',
                             '511 (-> -413 (-> -411 (-> 321 -211 -211) 22) -15 (-> -11 12 -16) 16)',
                             '-511 (-> 413 (-> 411 (-> -321 211 211) 22) 15 (-> 11 -12 16) -16)',
                             '511 (-> -413 (-> -411 (-> 321 -211 -211) 22) -15 (-> -13 14 -16) 16)',
                             '-511 (-> 413 (-> 411 (-> -321 211 211) 22) 15 (-> 13 -14 16) -16)']

mode_dict['sig_Dst_e_nu']=['511 (-> -413 (-> -411 (-> 321 -211 -211) 111) -11 12)',
                           '511 (-> -413 (-> -411 (-> 321 -211 -211) 22) -11 12)',
                           '-511 (-> 413 (-> 411 (-> -321 211 211) 111) 11 -12)',
                           '-511 (-> 413 (-> 411 (-> -321 211 211) 22) 11 -12)']

mode_dict['all_Dstst_tau_nu']=['511 (-> -10413 -15 (-> -11 12 -16) 16)','-511 (-> 10413 15 (-> 11 -12 16) -16)',
                               '511 (-> -10411 -15 (-> -11 12 -16) 16)','-511 (-> 10411 15 (-> 11 -12 16) -16)',
                               '511 (-> -20413 -15 (-> -11 12 -16) 16)','-511 (-> 20413 15 (-> 11 -12 16) -16)',
                               '511 (-> -415 -15 (-> -11 12 -16) 16)',  '-511 (-> 415 15 (-> 11 -12 16) -16)',
                               '521 (-> -10423 -15 (-> -11 12 -16) 16)','-521 (-> 10423 15 (-> 11 -12 16) -16)',
                               '521 (-> -10421 -15 (-> -11 12 -16) 16)','-521 (-> 10421 15 (-> 11 -12 16) -16)',
                               '521 (-> -20423 -15 (-> -11 12 -16) 16)','-521 (-> 20423 15 (-> 11 -12 16) -16)',
                               '521 (-> -425 -15 (-> -11 12 -16) 16)',  '-521 (-> 425 15 (-> 11 -12 16) -16)',
                               '511 (-> -10413 -15 (-> -13 14 -16) 16)','-511 (-> 10413 15 (-> 13 -14 16) -16)',
                               '511 (-> -10411 -15 (-> -13 14 -16) 16)','-511 (-> 10411 15 (-> 13 -14 16) -16)',
                               '511 (-> -20413 -15 (-> -13 14 -16) 16)','-511 (-> 20413 15 (-> 13 -14 16) -16)',
                               '511 (-> -415 -15 (-> -13 14 -16) 16)',  '-511 (-> 415 15 (-> 13 -14 16) -16)',
                               '521 (-> -10423 -15 (-> -13 14 -16) 16)','-521 (-> 10423 15 (-> 13 -14 16) -16)',
                               '521 (-> -10421 -15 (-> -13 14 -16) 16)','-521 (-> 10421 15 (-> 13 -14 16) -16)',
                               '521 (-> -20423 -15 (-> -13 14 -16) 16)','-521 (-> 20423 15 (-> 13 -14 16) -16)',
                               '521 (-> -425 -15 (-> -13 14 -16) 16)',  '-521 (-> 425 15 (-> 13 -14 16) -16)']

mode_dict['all_Dstst_e_nu']=['511 (-> -10413 -11 12)','-511 (-> 10413 11 -12)',
                         '511 (-> -10411 -11 12)','-511 (-> 10411 11 -12)',
                         '511 (-> -20413 -11 12)','-511 (-> 20413 11 -12)',
                         '511 (-> -415 -11 12)',  '-511 (-> 415 11 -12)',
                         '521 (-> -10423 -11 12)','-521 (-> 10423 11 -12)',
                         '521 (-> -10421 -11 12)','-521 (-> 10421 11 -12)',
                         '521 (-> -20423 -11 12)','-521 (-> 20423 11 -12)',
                         '521 (-> -425 -11 12)',  '-521 (-> 425 11 -12)',
                         '511 (-> -411 221 -11 12)','-511 (-> 411 221 11 -12)',
                         '511 (-> -411 111 -11 12)','-511 (-> 411 111 11 -12)',
                         '511 (-> -411 111 111 -11 12)','-511 (-> 411 111 111 11 -12)',
                         '511 (-> -411 211 -211 -11 12)','-511 (-> 411 211 -211 11 -12)',
                         '511 (-> -413 221 -11 12)','-511 (-> 413 221 11 -12)',
                         '511 (-> -413 111 -11 12)','-511 (-> 413 111 11 -12)',
                         '511 (-> -413 111 111 -11 12)','-511 (-> 413 111 111 11 -12)',
                         '511 (-> -413 211 -211 -11 12)','-511 (-> 413 211 -211 11 -12)',
                         '511 (-> -421 -211 -11 12)','-511 (-> 421 211 11 -12)',
                         '511 (-> -423 -211 -11 12)','-511 (-> 423 211 11 -12)']

mode_dict['sig_D_mu_nu']=['511 (-> -411 (-> 321 -211 -211) -13 14)',
                          '-511 (-> 411 (-> -321 211 211) 13 -14)']

mode_dict['sig_Dst_mu_nu']=['511 (-> -413 (-> -411 (-> 321 -211 -211) 111) -13 14)',
                           '511 (-> -413 (-> -411 (-> 321 -211 -211) 22) -13 14)',
                           '-511 (-> 413 (-> 411 (-> -321 211 211) 111) 13 -14)',
                           '-511 (-> 413 (-> 411 (-> -321 211 211) 22) 13 -14)']

mode_dict['all_Dstst_mu_nu']=['511 (-> -10413 -13 14)','-511 (-> 10413 13 -14)',
                          '511 (-> -10411 -13 14)','-511 (-> 10411 13 -14)',
                          '511 (-> -20413 -13 14)','-511 (-> 20413 13 -14)',
                          '511 (-> -415 -13 14)',  '-511 (-> 415 13 -14)',
                          '521 (-> -10423 -13 14)','-521 (-> 10423 13 -14)',
                          '521 (-> -10421 -13 14)','-521 (-> 10421 13 -14)',
                          '521 (-> -20423 -13 14)','-521 (-> 20423 13 -14)',
                          '521 (-> -425 -13 14)',  '-521 (-> 425 13 -14)',
                          '511 (-> -411 221 -13 14)','-511 (-> 411 221 13 -14)',
                          '511 (-> -411 111 -13 14)','-511 (-> 411 111 13 -14)',
                          '511 (-> -411 111 111 -13 14)','-511 (-> 411 111 111 13 -14)',
                          '511 (-> -411 211 -211 -13 14)','-511 (-> 411 211 -211 13 -14)',
                          '511 (-> -413 221 -13 14)','-511 (-> 413 221 13 -14)',
                          '511 (-> -413 111 -13 14)','-511 (-> 413 111 13 -14)',
                          '511 (-> -413 111 111 -13 14)','-511 (-> 413 111 111 13 -14)',
                          '511 (-> -413 211 -211 -13 14)','-511 (-> 413 211 -211 13 -14)',
                          '511 (-> -421 -211 -13 14)','-511 (-> 421 211 13 -14)',
                          '511 (-> -423 -211 -13 14)','-511 (-> 423 211 13 -14)']

In [77]:
# Apply 95% efficiency cuts to reduce data size
df_B_cut=df_B.query('D_vtxReChi2<13 and B0_vtxReChi2<14 and 5.03<B0_roeMbc_my_mask and -3.5<B0_roeDeltae_my_mask<0.5 and \
                4.65<B0_CMS2_weMbc and -2.2<B0_CMS0_weDeltae<0.5 and abs(B0_roeCharge_my_mask)<3 and \
                -3<B0_deltaE<-1 and e_CMS_p>0.2 and 0<=B0_vetoeID<0.9 and B0_vetomuID<0.9').copy()
df_B_cut['B0_isSignal'] = df_B_cut['B0_isSignal'].fillna(-1)
df_B_cut['D_isSignal'] = df_B_cut['D_isSignal'].fillna(-1)
df_B_cut['e_isSignal'] = df_B_cut['e_isSignal'].fillna(-1)
df_B_cut['B0_mcPDG'] = df_B_cut['B0_mcPDG'].fillna(0)

df_B_cut['DecayMode'] = df_B_cut.apply(decay_mode, axis=1)#.astype('category') #axis=0 will allow the application to be done at a column

df_B_cut.to_root('exp7_10_14b_e_bb_cut.root', key='B0')

In [78]:
df_all_cut = pandas.concat([df_B_cut,df_cont_cut],axis=0)
df_all_cut['DecayMode'] = df_all_cut['DecayMode'].astype('category')
df_all_cut.to_root('exp7_10_14b_e_all_cut.root', key='B0')
df_all_cut.columns

Index(['__experiment__', '__run__', '__event__', '__production__',
       '__candidate__', '__ncandidates__', '__weight__', 'B0_CMS_px',
       'B0_CMS_py', 'B0_CMS_pz',
       ...
       'e_p', 'e_E', 'e_isSignal', 'e_mcErrors', 'e_mcPDG', 'e_dM',
       'e_isBremsCorrected', 'e_genMotherPDG', 'e_nPXDHits', 'DecayMode'],
      dtype='object', length=210)

In [201]:
print(len(df_all_cut)==len(df_B_cut)+len(df_cont_cut))
df_all_cut.DecayMode.value_counts()

True


bkg                 190814
continuum            69620
all_Dstst_e_nu       48633
sig_D_e_nu           19411
sig_Dst_e_nu         17062
all_Dstst_mu_nu       5500
all_Dstst_tau_nu      2675
sig_D_tau_nu          1212
sig_Dst_tau_nu         751
sig_D_mu_nu            624
sig_Dst_mu_nu          375
Name: DecayMode, dtype: int64

## 3. Split Components

In [198]:
df_all_cut['isSignal'] = 0.0

In [199]:
# Signal components
sig_D_e_nu=df_all_cut.query('DecayMode=="sig_D_e_nu" and B0_mcErrors<16 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==511').copy()
sig_D_tau_nu=df_all_cut.query('DecayMode=="sig_D_tau_nu" and B0_mcErrors<32 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==15').copy()
sig_Dst_e_nu=df_all_cut.query('DecayMode=="sig_Dst_e_nu" and B0_mcErrors<64 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==511').copy()
sig_Dst_tau_nu=df_all_cut.query('DecayMode=="sig_Dst_tau_nu" and B0_mcErrors<64 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==15').copy()
all_Dstst_e_nu=df_all_cut.query('DecayMode=="all_Dstst_e_nu" and B0_mcErrors<64 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==511').copy()
all_Dstst_tau_nu=df_all_cut.query('DecayMode=="all_Dstst_tau_nu" and B0_mcErrors<64 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==15').copy()

#sig_D_mu_nu=df_all_cut.query('DecayMode=="sig_D_mu_nu" and B0_mcErrors<16').copy()
#sig_Dst_mu_nu=df_all_cut.query('DecayMode=="sig_Dst_mu_nu" and (16<=B0_mcErrors<32 or B0_mcErrors<8)').copy()
#all_Dstst_mu_nu=df_all_cut.query('DecayMode=="all_Dstst_mu_nu" and (16<=B0_mcErrors<64 or B0_mcErrors<8)').copy()

# Bkg components
bkg_fakeD = df_all_cut.query('abs(D_mcPDG)!=411 and B0_mcErrors!=512 and B0_isContinuumEvent!=1').copy()

bkg_combinatorial = df_all_cut.query('B0_mcPDG==300553 and abs(D_mcPDG)==411 and B0_mcErrors!=512 and B0_isContinuumEvent!=1').copy()

bkg_sigOtherBDTaudecay = df_all_cut.query('(DecayMode=="bkg" or DecayMode=="sig_D_mu_nu" or DecayMode=="sig_Dst_mu_nu" or DecayMode=="all_Dstst_mu_nu") and \
B0_mcPDG!=300553 and abs(D_mcPDG)==411 and B0_mcErrors!=512 and B0_isContinuumEvent!=1').copy()

bkg_recoFakeTracksClusters = df_all_cut.query('B0_mcErrors==512 and B0_isContinuumEvent!=1').copy()
bkg_continuum = df_all_cut.query('B0_isContinuumEvent==1').copy()

bkg_others = pandas.concat([df_all_cut,
                           sig_D_e_nu,
                           sig_D_tau_nu,
                           sig_Dst_e_nu,
                           sig_Dst_tau_nu,
                           all_Dstst_e_nu,
                           all_Dstst_tau_nu,
                           bkg_fakeD,
                           bkg_combinatorial,
                           bkg_sigOtherBDTaudecay,
                           bkg_recoFakeTracksClusters,
                           bkg_continuum]).drop_duplicates(keep=False)
# Weird! the bkg_others contains some events with
# correct sig decay hash chain and correct B0_mcPDG, D_mcPDG, e_genMotherPDG,
# but with 128< B0_mcErrors < 256 (misID)

In [200]:
# check duplicate, this boolean Series should all False
check_duplicate = pandas.concat([sig_D_e_nu,
                                 sig_D_tau_nu,
                                 sig_Dst_e_nu,
                                 sig_Dst_tau_nu,
                                 all_Dstst_e_nu,
                                 all_Dstst_tau_nu,
                                 bkg_fakeD,
                                 bkg_combinatorial,
                                 bkg_sigOtherBDTaudecay,
                                 bkg_recoFakeTracksClusters,
                                 bkg_continuum]).duplicated()
print(np.where(check_duplicate)[0]) # returns an array of indices of the True entries, should be empty
len(bkg_fakeD)+len(sig_D_e_nu)+len(sig_D_tau_nu)+len(sig_Dst_e_nu)+len(sig_Dst_tau_nu)+len(all_Dstst_e_nu)+len(all_Dstst_tau_nu)+len(bkg_combinatorial)+len(bkg_sigOtherBDTaudecay)+len(bkg_recoFakeTracksClusters)+len(bkg_continuum)+len(bkg_others)==len(df_all_cut)

[]


True

In [202]:
# add isSignal column for signal components
sig_D_e_nu['isSignal']=1.0
sig_D_tau_nu['isSignal']=1.0
sig_Dst_e_nu['isSignal']=1.0
sig_Dst_tau_nu['isSignal']=1.0
all_Dstst_e_nu['isSignal']=1.0
all_Dstst_tau_nu['isSignal']=1.0

df_all_cut2 = pandas.concat([sig_D_e_nu,
                             sig_D_tau_nu,
                             sig_Dst_e_nu,
                             sig_Dst_tau_nu,
                             all_Dstst_e_nu,
                             all_Dstst_tau_nu,
                             bkg_fakeD,
                             bkg_combinatorial,
                             bkg_sigOtherBDTaudecay,
                             bkg_recoFakeTracksClusters,
                             bkg_continuum,
                             bkg_others])

df_all_cut2.eval('B_D_ReChi2 = B0_vtxReChi2 + D_vtxReChi2', inplace=True)
df_all_cut2.eval('p_D_l = D_CMS_p + e_CMS_p', inplace=True)
df_all_cut2.to_root('exp7_10_14b_e_all_cut.root', key='B0')
df_all_cut2.isSignal.value_counts()

0.0    318384
1.0     38293
Name: isSignal, dtype: int64

In [ ]:
pandas.set_option('display.max_rows', None)
bkg_sigOtherBDTaudecay.B0_DplusMode.value_counts()

In [ ]:
cut='B0_roeMbc_my_mask>5.2 and abs(B0_mcPDG)!=521'
candidate12 = bkg_others.query(cut).iloc[6][['B0_DecayHash', "B0_DecayHashEx"]].values

# print the original decay as simulated in MC with removed Bremsstrahlung gammas
print("Monte Carlo Decay with removed Bremsstrahlung gammas: ")
org2 = hashmap2.get_original_decay(*candidate12)
print(org2.to_string())

# 4. Write out new Ntuple
This is only needed if going to be used as bkg training sample

sig_D_tau_nu.to_root('exp7_10_14b_e_D_tau_nu.root', key='B0')

sig_D_e_nu.to_root('exp7_10_14b_e_other_signal.root', key='B0')
sig_Dst_e_nu.to_root('exp7_10_14b_e_other_signal.root', key='B0',mode='a')
sig_Dst_tau_nu.to_root('exp7_10_14b_e_other_signal.root', key='B0',mode='a')
all_Dstst_e_nu.to_root('exp7_10_14b_e_other_signal.root', key='B0',mode='a')
all_Dstst_tau_nu.to_root('exp7_10_14b_e_other_signal.root', key='B0',mode='a')

bkg_fakeD.to_root('exp7_10_14b_e_bbkg1_fakeD.root', key='B0')
bkg_combinatorial.to_root('exp7_10_14b_e_bbkg2.root', key='B0')
bkg_sigOtherBDTaudecay.to_root('exp7_10_14b_e_bbkg2.root', key='B0',mode='a')
bkg_others.to_root('exp7_10_14b_e_bbkg2.root', key='B0',mode='a')
bkg_recoFakeTracksClusters.to_root('exp7_10_14b_e_bbkg2.root', key='B0',mode='a')
#bkg_continuum.to_root('exp7_10_14b_e_continuum.root', key='B0')

## ROE Ntuple
Check the ROE object kinematics

signal_event_list=df_cut.query('(DecayMode=="D_tau_nu" or DecayMode=="D_e_nu") and abs(B0_mcPDG)==511').__event__.unique()
print(type(signal_event_list))
df_pi_signal=df_pi[df_pi.__event__.isin(signal_event_list)]
df_gamma_signal=df_gamma.query('__event__ in @signal_event_list')
#df_pi.query('__event__ in @signal_event_list')
df_pi_others=df_pi[~df_pi.__event__.isin(signal_event_list)]
df_gamma_others=df_gamma.query('__event__ not in @signal_event_list')
#df_pi.query('__event__ not in @signal_event_list')

df_pi_signal.to_root('exp7_14_e_ROE.root', key='pi_roe_signal')
df_gamma_signal.to_root('exp7_14_e_ROE.root', key='gamma_roe_signal',mode='a')
df_pi_others.to_root('exp7_14_e_ROE.root', key='pi_roe_others',mode='a')
df_gamma_others.to_root('exp7_14_e_ROE.root', key='gamma_roe_others',mode='a')

# 5. Apply MVAs

In [203]:
import basf2_mva

df_startMVA = df_all_cut2
identifier_1 = '/home/belle/zhangboy/R_D/Generic_MC14rd/Continuum_Suppression/MVA1_FastBDT.xml'
test_1 = 'exp7_10_14b_e_all_cut.root'
output_file_1 = 'exp7_10_14b_e_all_cut_MVA1.root'

identifier_1_5 = '/home/belle/zhangboy/R_D/Generic_MC14rd/B_bkg_Suppression/MVA1_5/MVA1_5_FastBDT.xml'
test_1_5 = 'exp7_10_14b_e_all_cut.root'
output_file_1_5 = 'exp7_10_14b_e_all_cut_MVA1_5.root'

output_file_1_5_applied = 'exp7_10_14b_e_all_cut_MVA1_5_applied.root'

identifier_2_1 = '/home/belle/zhangboy/R_D/Generic_MC14rd/B_bkg_Suppression/MVA2/MVA2_1_FastBDT.xml'
test_2_1 = output_file_1_5_applied
output_file_2_1 = 'exp7_10_14b_e_all_cut_MVA2_1.root'
output_file_2_1_applied = 'exp7_10_14b_e_all_cut_MVA2_1_applied.root'

In [204]:
# apply CS BDT identifier_1, merge data file and mva output, rename the column
basf2_mva.expert(basf2_mva.vector(identifier_1),  # weightfile
                 basf2_mva.vector(test_1),
                 'B0', output_file_1)

df1 = df_startMVA.drop_duplicates(subset=['__experiment__','__run__','__event__','__production__','__candidate__']).reset_index(drop=True)
df2 = root_pandas.read_root(output_file_1)
print(len(df1)==len(df2))
df_1 = pandas.concat([df1,df2],axis=1)

df_1=df_1.rename(columns={"__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slContinuum_Suppression__slMVA1_FastBDT__ptxml": "MVA1_output"})
df_1=df_1.drop(columns=['__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slContinuum_Suppression__slMVA1_FastBDT__ptxml_isSignal'])

# apply BDT 1_5 identifier_1_5, merge, rename, change the output type, save
basf2_mva.expert(basf2_mva.vector(identifier_1_5),  # weightfile
                 basf2_mva.vector(test_1_5),
                 'B0', output_file_1_5)

df3 = root_pandas.read_root(output_file_1_5)
print(len(df_1)==len(df3))
df_1_5 = pandas.concat([df_1,df3],axis=1)

df_1_5=df_1_5.rename(columns={"__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slB_bkg_Suppression__slMVA1_5__slMVA1_5_FastBDT__ptxml": "MVA1_5_output"})
df_1_5=df_1_5.drop(columns=['__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slB_bkg_Suppression__slMVA1_5__slMVA1_5_FastBDT__ptxml_isSignal'])

df_1_5.MVA1_5_output=np.float64(df_1_5.MVA1_5_output)
print(type(df_1_5.MVA1_5_output[0]))
print(type(df_1_5.isSignal[0]))

df_1_5.to_root(output_file_1_5_applied, key='B0')

# apply BDT 2_1 identifier_2_1, merge, rename, save
basf2_mva.expert(basf2_mva.vector(identifier_2_1),  # weightfile
                 basf2_mva.vector(test_2_1),
                 'B0', output_file_2_1)

df4 = root_pandas.read_root(output_file_2_1)
print(len(df_1_5)==len(df4))
df_2_1 = pandas.concat([df_1_5, df4],axis=1)
print(len(df_1_5)==len(df_2_1))

df_2_1=df_2_1.rename(columns={"__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slB_bkg_Suppression__slMVA2__slMVA2_1_FastBDT__ptxml": "MVA2_1_output"})
df_2_1=df_2_1.drop(columns=['__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slB_bkg_Suppression__slMVA2__slMVA2_1_FastBDT__ptxml_isSignal'])

df_2_1.to_root(output_file_2_1_applied, key='B0')

[INFO] Elapsed application time in ms 6009.73 for MVA1_FastBDT.xml
[WARNING] String passed to makeROOTCompatible contains double-underscore __, which is used internally for escaping special characters. It is recommended to avoid this. However escaping a string twice with makeROOTCompatible is safe, but will print this warning. Passed string: __slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slContinuum_Suppression__slMVA1_FastBDT__ptxml_isSignal
True
[INFO] Elapsed application time in ms 3923.46 for MVA1_5_FastBDT.xml
[WARNING] String passed to makeROOTCompatible contains double-underscore __, which is used internally for escaping special characters. It is recommended to avoid this. However escaping a string twice with makeROOTCompatible is safe, but will print this warning. Passed string: __slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slB_bkg_Suppression__slMVA1_5__slMVA1_5_FastBDT__ptxml_isSignal
True
<class 'numpy.float64'>
<class 'numpy.float64'>
[INFO] Elapsed appli

In [205]:
print(df_2_1.columns)
len(df_2_1)

Index(['__experiment__', '__run__', '__event__', '__production__',
       '__candidate__', '__ncandidates__', '__weight__', 'B0_CMS_px',
       'B0_CMS_py', 'B0_CMS_pz',
       ...
       'e_mcPDG', 'e_dM', 'e_isBremsCorrected', 'e_genMotherPDG', 'e_nPXDHits',
       'DecayMode', 'isSignal', 'MVA1_output', 'MVA1_5_output',
       'MVA2_1_output'],
      dtype='object', length=214)


356677